# 07 — Vector DB (Chroma) + Relational DB (Postgres) Together
Pattern: Chroma holds embeddings for similarity search; Postgres holds structured metadata, full document text, and relationships (users, permissions, versions). Query flow: vector search in Chroma → get ids → fetch full rows from Postgres by id.

**Why split them at all:** vector DBs are great at nearest-neighbor search but weak at relational joins/filters/transactions. Postgres is the opposite. Most real systems use both rather than forcing one to do the other's job.

**Corrected in this version:** `InHouseEmbeddings(model=MODEL_JINA)` → `InHouseEmbeddings()` — the corrected class takes no `model=` kwarg.

# Setup
Run this first in every notebook. It assumes this notebook lives in a folder
that can reach `inhouse_wrappers.py` (the CORRECTED version, from `wrapper_fix/`),
`rag_pure_python.py`, and `inhouse_llm.py`. Adjust the `sys.path.append(...)`
lines below if your folder layout differs.

**Corrected in this version:** uses `ask()`/`ask_vision()` (built on the fixed
`get_chat_model()`) instead of calling `multimodal_chat()` directly — the
original always hit the Qwen3-14B endpoint regardless of which `model=` you
asked for. Embeddings go through `embedder.embed_query()`/`.embed_documents()`
instead of `get_embedding(text, model=MODEL_JINA)`, which doesn't match the
real function signature in your `inhouse_llm.py` (no `model=` kwarg there).

In [ ]:
import sys, os
sys.path.append(os.path.abspath("../wrapper_fix"))           # folder containing the corrected inhouse_wrappers.py
sys.path.append(os.path.abspath("."))                          # folder containing inhouse_llm.py / rag_pure_python.py
# sys.path.append("/path/to/inhouse_rag_capstone")              # uncomment & adjust if needed

from inhouse_llm import MODEL_QWEN3_14B, MODEL_QWEN3_30B, MODEL_MISTRAL, MODEL_LLAMA, MODEL_DEVSTRAL, MODEL_QWEN2_5_VL_7B, MODEL_JINA
from inhouse_wrappers import get_chat_model, InHouseEmbeddings, build_vision_messages, llm_for
from langchain_core.messages import SystemMessage, HumanMessage
from rag_pure_python import chunk_text, SimpleVectorStore, generate_answer

embedder = InHouseEmbeddings()

def ask(system_prompt, user_prompt, model=MODEL_QWEN3_14B, max_tokens=500):
    """Correctly-routed replacement for calling multimodal_chat() directly."""
    llm = get_chat_model(model=model, max_tokens=max_tokens)
    return llm.invoke([SystemMessage(content=system_prompt), HumanMessage(content=user_prompt)]).content

def ask_vision(system_prompt, user_prompt, image_base64, model=MODEL_QWEN2_5_VL_7B, max_tokens=500):
    """Correctly-routed, correctly-formatted multimodal call."""
    llm = get_chat_model(model=model, max_tokens=max_tokens)
    return llm.invoke(build_vision_messages(system_prompt, user_prompt, image_base64)).content

print("Setup OK")

## 1. Chroma setup (vector layer)
Recall: always pass `embedding_function=InHouseEmbeddings()` explicitly so Chroma never falls back to downloading a default HF embedding model.

In [ ]:
# pip install chromadb --break-system-packages
import chromadb

chroma_client = chromadb.PersistentClient(path="./chroma_db_capstone")

collection = chroma_client.get_or_create_collection(name="capstone_docs")

docs = [
    "MCP standardizes how LLMs call external tools through a client-server interface.",
    "RAG combines a retriever and a generator to ground LLM answers in retrieved context.",
    "Qwen3-14B is the default chat model for general RAG generation tasks.",
]
ids = ["doc_1", "doc_2", "doc_3"]
embeddings = embedder.embed_documents(docs)

collection.add(ids=ids, embeddings=embeddings, documents=docs)
print("Chroma collection count:", collection.count())

## 2. Postgres setup (relational layer)
We store the SAME ids alongside richer metadata (owner, created_at, version, tags) that Chroma either can't model well or you don't want duplicated/out-of-sync in two places.

Requires a running Postgres instance. `pip install psycopg2-binary --break-system-packages`. Adjust connection params for your environment.

In [ ]:
import psycopg2

conn = psycopg2.connect(
    host="localhost", port=5432, dbname="capstone", user="postgres", password="postgres"
)
conn.autocommit = True
cur = conn.cursor()

cur.execute("""
CREATE TABLE IF NOT EXISTS documents (
    id TEXT PRIMARY KEY,
    content TEXT NOT NULL,
    owner TEXT,
    tags TEXT[],
    created_at TIMESTAMP DEFAULT now()
);
""")

rows = [
    ("doc_1", docs[0], "team_a", ["mcp", "protocol"]),
    ("doc_2", docs[1], "team_a", ["rag", "core-concept"]),
    ("doc_3", docs[2], "team_b", ["model-selection"]),
]
for r in rows:
    cur.execute(
        "INSERT INTO documents (id, content, owner, tags) VALUES (%s, %s, %s, %s) "
        "ON CONFLICT (id) DO NOTHING;", r
    )
print("Postgres rows inserted")

## 3. The joined query pattern
1. Embed the query, search Chroma for nearest ids.
2. Take those ids to Postgres for full metadata, permission filtering, or audit logging.
This is the standard 'vector DB for search, relational DB for everything else' shape.

In [ ]:
def search_and_join(query, k=2, owner_filter=None):
    q_vec = embedder.embed_query(query)
    results = collection.query(query_embeddings=[q_vec], n_results=k)
    matched_ids = results["ids"][0]

    cur.execute(
        "SELECT id, content, owner, tags FROM documents WHERE id = ANY(%s)"
        + (" AND owner = %s" if owner_filter else "") + ";",
        (matched_ids, owner_filter) if owner_filter else (matched_ids,)
    )
    return cur.fetchall()

for row in search_and_join("How do LLMs call external tools?"):
    print(row)

print("\n-- filtered to team_a only --")
for row in search_and_join("model selection", owner_filter="team_a"):
    print(row)

## 4. Optional: pgvector instead of two databases
If you'd rather not run two systems, Postgres's `pgvector` extension stores embeddings as a native column type and lets you do nearest-neighbor search *inside* the same SQL query as your relational filters — one database, one transaction, simpler ops. Trade-off: Chroma/dedicated vector DBs are usually faster at large-scale nearest-neighbor search; pgvector is simpler when your corpus is small/medium and you want everything transactionally consistent.

In [ ]:
# Requires: CREATE EXTENSION IF NOT EXISTS vector;  (run once, needs superuser)
cur.execute("CREATE EXTENSION IF NOT EXISTS vector;")
cur.execute("""
CREATE TABLE IF NOT EXISTS documents_with_vectors (
    id TEXT PRIMARY KEY,
    content TEXT NOT NULL,
    owner TEXT,
    embedding vector(1024)  -- match your Jina embedding dimension
);
""")

for doc_id, content, vec in zip(ids, docs, embeddings):
    cur.execute(
        "INSERT INTO documents_with_vectors (id, content, embedding) VALUES (%s, %s, %s) "
        "ON CONFLICT (id) DO NOTHING;",
        (doc_id, content, vec)
    )

query_vec = embedder.embed_query("How do LLMs call external tools?")
cur.execute(
    "SELECT id, content, embedding <-> %s::vector AS distance "
    "FROM documents_with_vectors ORDER BY distance LIMIT 2;",
    (query_vec,)
)
print(cur.fetchall())

### When to use which
- **Chroma (or FAISS/Milvus/etc.) + Postgres separately**: large corpora, need a vector-DB-specific feature (e.g., HNSW tuning, filtering syntax), or vector and relational data are owned by different services.
- **pgvector inside Postgres**: smaller/medium corpora, you want one system to operate and back up, and you value transactional consistency between metadata and vectors over raw vector-search throughput.

For a capstone, pgvector is often the simpler choice to demo end-to-end; Chroma is worth knowing because it's what you'll most often see in tutorials and other teams' code.